## Shared setup

In [ ]:
# %pip install facenet-pytorch tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 140.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 111.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56

In [1]:
import os
import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw
from pathlib import Path
from tqdm import tqdm
from facenet_pytorch import MTCNN, InceptionResnetV1
from google.colab import drive

In [2]:
# --- Mount Google Drive ---
drive.mount("/content/drive")

# --- Folder Setup ---
IMAGES_DIR = Path("/content/all_images")                                    # unzipped images
DRIVE_SAVE = Path("/content/drive/MyDrive/walk-of-life/face_recognition")   # .pt (embeddings) and .pkl (metadata) save location
EMBEDDINGS_FILE = DRIVE_SAVE / "embeddings_db.pt" 

DRIVE_SAVE.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


##### Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### load dataset

In [ ]:
!cp "/content/drive/MyDrive/walk-of-life/dataset1.zip" "/content/"
!cp "/content/drive/MyDrive/walk-of-life/dataset2.zip" "/content/"
!cp "/content/drive/MyDrive/walk-of-life/dataset3.zip" "/content/"

In [ ]:
!unzip -q /content/dataset1.zip -d /content/all_images/
!unzip -q /content/dataset2.zip -d /content/all_images/
!unzip -q /content/dataset3.zip -d /content/all_images/

## Part 1: extract emdedding for later identification and save results on drive

### define embedding database

In [ ]:
def build_embedding_database(
    images_dir: Path,
    save_path: Path,
    batch_size: int = 16,
    image_size: int = 160,
    min_face_prob: float = 0.90,
):
    """
    Iterate every image in `images_dir`, detect all faces with MTCNN,
    extract 512-d embeddings via InceptionResnetV1, and persist the
    result to `save_path` as a .pt file.

    Saved structure (list of dicts):
      [
        {
          "filename":  str,           # e.g. "IMG_4821.jpg"
          "bbox":      list[float],   # [x1, y1, x2, y2] in original pixels
          "prob":      float,         # MTCNN detection confidence
          "embedding": Tensor(512,),  # L2-normalised, on CPU
        },
        ...
      ]
    """

    # --- Models ---
    # keep_all=True  → return every detected face, not just the best one
    # post_process=True → normalise pixel values to [-1, 1] for the resnet
    mtcnn = MTCNN(
        image_size=image_size,
        keep_all=True,
        post_process=True,
        device=device,
        min_face_size=40,          # ignore very small/blurry faces
        thresholds=[0.6, 0.7, 0.7] # MTCNN stage thresholds (P/R/O-net)
    )

    resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

    # --- Collect image paths ---
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = [
        p for p in sorted(images_dir.iterdir())
        if p.suffix.lower() in valid_ext
    ]
    print(f"Found {len(image_paths):,} images in {images_dir}")

    records = []
    skipped = 0

    for img_path in tqdm(image_paths, desc="Extracting embeddings", unit="img"):
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            tqdm.write(f"  [SKIP] Cannot open {img_path.name}: {e}")
            skipped += 1
            continue

        # --- Detect faces ---
        # boxes  : Tensor[N, 4] or None
        # probs  : Tensor[N]    or None
        # faces  : Tensor[N, 3, H, W] or None   (aligned & cropped)
        try:
            boxes, probs, faces = mtcnn.detect(img, landmarks=False), None, None
            # detect() only returns boxes+probs; forward() returns cropped faces
            faces = mtcnn(img)          # [N, 3, 160, 160] or None
            boxes, probs = mtcnn.detect(img)
        except Exception as e:
            tqdm.write(f"  [SKIP] MTCNN error on {img_path.name}: {e}")
            skipped += 1
            continue

        if faces is None or boxes is None:
            continue  # no faces detected in this image

        # Filter by confidence
        keep = probs >= min_face_prob
        if not keep.any():
            continue

        faces  = faces[keep]     # [K, 3, 160, 160]
        boxes  = boxes[keep]     # [K, 4]
        probs  = probs[keep]     # [K]

        # --- Extract embeddings ---
        with torch.no_grad():
            face_batch = faces.to(device)               # [K, 3, 160, 160]
            embeddings = resnet(face_batch)              # [K, 512]  (L2-normed by default)
            embeddings = embeddings.cpu()

        # --- Store one record per detected face ---
        for i in range(len(embeddings)):
            records.append({
                "filename":  img_path.name,
                "bbox":      boxes[i].tolist(),          # [x1, y1, x2, y2]
                "prob":      float(probs[i]),
                "embedding": embeddings[i],              # Tensor(512,)
            })

    print(f"\nDone. {len(records):,} face records from "
          f"{len(image_paths) - skipped:,} images "
          f"({skipped} skipped).")
 
    # --- Save to Drive ---
    torch.save(records, save_path)
    print(f"Saved → {save_path}  ({save_path.stat().st_size / 1e6:.1f} MB)")
    return records


#### run embedding extraction

In [ ]:
records = build_embedding_database(IMAGES_DIR, EMBEDDINGS_FILE)

NameError: name 'build_embedding_database' is not defined

## Part 2: Search
